# 02 Symbol Optimize - MA Cross

Search MA Cross parameter candidates for one symbol, filter the shortlist, and create a clean selected-parameter handoff for later validation.


In [ ]:
# Cell 1 - Safe import path bootstrap

import sys
from pathlib import Path


def find_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'pyproject.toml').exists() and (p / 'core_python' / 'shared').exists():
            return p
    raise RuntimeError('Cannot find SEN05 repo root')


ROOT = find_root(Path.cwd())
CORE = ROOT / 'core_python'
for p in (str(ROOT), str(CORE)):
    if p not in sys.path:
        sys.path.insert(0, p)

print('ROOT =', ROOT)


In [ ]:
# Cell 2 - Imports and notebook setup

from IPython.display import display

from core_python.strategies.ma_cross.research_utils import (
    configure_notebook,
    export_research_bundle,
    plot_optimization_dashboard,
    run_symbol_grid_search,
    select_top_candidates,
    show_best_candidate,
    show_note,
    show_optimizer_filter_report,
    show_run_config,
    show_selected_params,
    show_strategy_summary,
)

configure_notebook()
show_strategy_summary()


In [ ]:
# Cell 3 - Optimization configuration

RUN_CONFIG = {
    'symbol': 'US30',
    'account_mode': 'standard',
    'initial_balance': 100_000.0,
    'date_from': '2022-01-01',
    'date_to': None,
    'max_bars': 50_000,
    'broker_profile': None,
    'costs': {},
    'search_space': {
        'fast_ma': [8, 10, 12],
        'slow_ma': [18, 20, 24, 30],
        'atr_stop_mult': [1.5, 2.0, 2.5],
        'atr_tp_mult': [0.0, 2.0, 3.0],
        'timeframe': ['M20', 'M30', 'M45'],
        'ma_type': ['sma', 'ema'],
    },
    'export_report': False,
}
FILTER = {
    'top_n': 15,
    'min_trades': 40,
    'min_profit_factor': 1.10,
    'max_drawdown': 25.0,
    'score_column': 'sharpe',
}

show_run_config('MA Cross Optimize Configuration', RUN_CONFIG)
show_run_config('Candidate Gates', FILTER)


In [ ]:
# Cell 4 - Run grid search

grid = run_symbol_grid_search(**{k: v for k, v in RUN_CONFIG.items() if k != 'export_report'})
show_note('Grid Search Results', 'This table is a shortlist source, not final evidence. Validate selected candidates out of sample.')
display(grid.head(30))
plot_optimization_dashboard(grid, score_col=FILTER['score_column'])


In [ ]:
# Cell 5 - Filter and select candidates

candidates = select_top_candidates(grid, **FILTER)
show_optimizer_filter_report(grid, candidates, FILTER)
display(candidates)

best = show_best_candidate(candidates)
best_params = best.to_dict() if best is not None else None
show_selected_params(best_params, title='Selected MA Cross Parameters')
best_params


In [ ]:
# Cell 6 - Optional export

if RUN_CONFIG.get('export_report'):
    export_path = export_research_bundle(
        {'grid': grid, 'candidates': candidates},
        name=f"{RUN_CONFIG['symbol']}_ma_cross_optimize",
    )
    print('Exported:', export_path)
else:
    print("Export is disabled. Set RUN_CONFIG['export_report'] = True to save CSV files.")
